## Manuscript-ready plots and tables


In [ ]:
import sys
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

# This notebook lives in notebooks/expanded/, so the repo root is two levels up.
REPO_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
sys.path.insert(0, str(Path.cwd()))

from exp_plot_utils import (
    _configure_fonts,
    SCENARIO_EXTENDED_MAP,
    OKABE_ITO,
    GRAPH_COLOR,
)

_configure_fonts()

In [ ]:
result_path = REPO_ROOT / "data" / "analysis" / "aggregated_expanded"

AGENT_COLS = ["agent_plasticity", "agent_monotonicity", "agent_influence_proj_contrast"]
RUN_COLS = ["pop_consensus_change"]

COL_PRETTY_NAMES = {
    "agent_plasticity": "Plasticity",
    "agent_monotonicity": "Directedness",
    "agent_influence_proj_contrast": "Outgoing Influence",
    "pop_consensus_change": "Consensus Change",
}
OUTCOME_ORDER = [COL_PRETTY_NAMES[o] for o in AGENT_COLS + RUN_COLS]

contrast_labels = {
    "role_eff": "Role effect",
    "model_eff": "Specialization effect",
    "alignment": "Role--spec. alignment",
    "dissociation": "Composition effect",
}
CONTRAST_ORDER = list(contrast_labels.values())

graph_labels = {
    "erdos-renyi": "ER",
    "watts-strogatz": "WS",
    "marginal": "Marginal",
}


def fmt_p(p: float) -> str:
    if pd.isna(p):
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def col_decimals(values, base: int = 2, max_nd: int = 4) -> int:
    """Fewest decimals >= base such that no nonzero value prints as zero (e.g. 0.003 -> 3, not 0.00)."""
    v = pd.Series(np.asarray(values, dtype=float).ravel()).dropna()
    v = v[v != 0].abs()
    for nd in range(base, max_nd + 1):
        if (v.round(nd) > 0).all():
            return nd
    return max_nd


def fmt_cols(
    df: pd.DataFrame, cols: list[str], base: int = 2, by: str = "Outcome"
) -> None:
    """Format `cols` in place as strings, with one shared number of decimals per `by` group
    (an index level or a column; outcomes differ in scale), chosen by `col_decimals`."""
    keys = df.index.get_level_values(by) if by in df.index.names else df[by]
    keys = np.asarray(keys)
    out = {c: df[c].to_numpy(dtype=object, copy=True) for c in cols}
    for key in pd.unique(keys):
        mask = keys == key
        nd = col_decimals(df.loc[mask, cols].to_numpy(), base)
        for c in cols:
            out[c][mask] = [
                "" if pd.isna(x) else f"{x:.{nd}f}" for x in df.loc[mask, c]
            ]
    for c in cols:
        df[c] = out[c]


def latex_cols(df: pd.DataFrame) -> str:
    """Left-aligned index columns, right-aligned values (strings would otherwise be left-aligned)."""
    return "l" * df.index.nlevels + "r" * df.shape[1]


def fmt_contrast_cols(df: pd.DataFrame) -> pd.DataFrame:
    """d and its CI share decimals; SE gets 3+; t gets 2+."""
    df = df.copy()
    fmt_cols(df, ["Cohen's d", "d CI lower", "d CI upper"], base=2)
    fmt_cols(df, ["SE"], base=3)
    fmt_cols(df, ["t"], base=2)
    return df


## Aggregated stats

In [ ]:
EMM_COLS = [
    "run_scenario",
    "emmean",
    "SE",
    "df",
    "lower.CL",
    "upper.CL",
    "graph_family",
    "metric",
]

agg_res = []
for col in AGENT_COLS + RUN_COLS:
    level = "runs" if col in RUN_COLS else "agent"
    r = (
        pl.read_csv(result_path / "aggregated" / level / f"{col}_scenario" / "emm.csv")
        .with_columns(
            pl.lit("marginal").alias("graph_family"), pl.lit(col).alias("metric")
        )
        .select(EMM_COLS)
    )
    agg_res.append(r)
    r = (
        pl.read_csv(
            result_path / "aggregated" / level / f"{col}_scenario_family" / "emm.csv"
        )
        .with_columns(pl.lit(col).alias("metric"))
        .select(EMM_COLS)
    )
    agg_res.append(r)

lfs_res = (
    pl.concat(agg_res)
    .rename(
        {
            "run_scenario": "Scenario",
            "emmean": "Estimate",
            "lower.CL": "CI low",
            "upper.CL": "CI high",
            "metric": "Outcome",
            "graph_family": "Network",
        }
    )
    .with_columns(
        pl.col("Outcome").map_elements(
            lambda x: COL_PRETTY_NAMES.get(x, x), return_dtype=pl.String
        ),
        pl.col("Scenario").map_elements(
            lambda x: SCENARIO_EXTENDED_MAP.get(x, x), return_dtype=pl.String
        ),
        pl.col("Network").map_elements(
            lambda x: graph_labels.get(x, x), return_dtype=pl.String
        ),
    )
    .select(
        "Scenario",
        "Outcome",
        "Network",
        "Estimate",
        "CI low",
        "CI high",
        "SE",
        "df",
    )
).to_pandas()

lfs_res["Scenario"] = pd.Categorical(
    lfs_res["Scenario"], categories=SCENARIO_EXTENDED_MAP.values(), ordered=True
)
lfs_res["Outcome"] = pd.Categorical(
    lfs_res["Outcome"], categories=OUTCOME_ORDER, ordered=True
)
lfs_res["Network"] = pd.Categorical(
    lfs_res["Network"], categories=["ER", "WS", "Marginal"], ordered=True
)
lfs_res.to_csv(result_path / "aggregated" / "agg_results.csv", index=False)

lfs_res = lfs_res.sort_values(["Outcome", "Scenario", "Network"]).set_index(
    ["Outcome", "Scenario", "Network"]
)
lfs_res

### LaTeX

In [ ]:
tex_res = lfs_res.copy()
fmt_cols(tex_res, ["Estimate", "CI low", "CI high"], base=2)
fmt_cols(tex_res, ["SE"], base=3)
tex_res["df"] = tex_res["df"].round().astype(int)

out_res = tex_res.to_latex(
    index=True,
    column_format=latex_cols(tex_res),
    caption="",
    multicolumn=True,
    multirow=True,
    escape=False,
    label="tab:agg-results",
)
out_res = out_res.replace("\\multirow[t]{", "\\multirow[c]{")
print(out_res)


## Contrasts

In [ ]:
agent_contrast = pd.read_csv(result_path / "contrasts" / "contrast_summary_agent.csv")
run_contrast = pd.read_csv(
    result_path / "contrasts" / "runs" / "contrast_summary_runs.csv"
)


contrasts = pd.concat([agent_contrast, run_contrast], ignore_index=True)
contrasts = contrasts[contrasts["outcome"].isin(COL_PRETTY_NAMES.keys())]
contrasts.to_csv(result_path / "contrasts" / "combined_contrasts.csv", index=False)
contrasts.sample(10, random_state=0)

In [ ]:
def plot_contrast_forest(
    summary,
    contrasts_order=None,
    outcomes_order=None,
    graph_families_order=("erdos-renyi", "watts-strogatz"),
    use_standardized=True,
    figsize=(14, 4.8),
):
    """Forest plot with one panel per contrast and 3 estimates per outcome.

    For each outcome row inside each contrast panel, this plots a triplet of
    estimates with CIs:
      - Marginal (pooled)
      - Within Erdos-Renyi
      - Within Watts-Strogatz

    Args:
        summary: DataFrame from build_contrast_summary.
        contrasts_order: Ordered list of contrasts to display as panels.
        outcomes_order: Ordered list of outcomes to display on y-axis.
        graph_families_order: Order of graph families used for within contrasts.
        use_standardized: Whether to use standardized estimates/CIs.
        figsize: Matplotlib figure size.

    Returns:
        Matplotlib figure.
    """
    if contrasts_order is None:
        contrasts_order = ["role_eff", "model_eff", "alignment", "dissociation"]
    if outcomes_order is None:
        outcomes_order = summary["outcome"].unique().tolist()

    contrast_labels = {
        "role_eff": "Role Effect",
        "model_eff": "Specialization Effect",
        "alignment": "Role-Specialization Effect",
        "dissociation": "Composition Effect",
    }

    triplet_specs = [
        ("marginal", None, "Marginal (pooled)", OKABE_ITO["blue"], 0.0),
        (
            "within",
            graph_families_order[0],
            "Erdos-Renyi",
            GRAPH_COLOR["erdos-renyi"],
            -0.23,
        ),
        (
            "within",
            graph_families_order[1],
            "Watts-Strogatz",
            GRAPH_COLOR["watts-strogatz"],
            0.23,
        ),
    ]

    est_col = "cohens_d" if use_standardized else "estimate"
    lo_col = "cohens_d_lower" if use_standardized else "lower"
    hi_col = "cohens_d_upper" if use_standardized else "upper"

    fig, axes = plt.subplots(
        1,
        len(contrasts_order),
        figsize=figsize,
        sharey=True,
        sharex=True,
        squeeze=False,
    )
    axes = axes[0]

    y_positions = {o: i for i, o in enumerate(outcomes_order)}

    for panel_idx, (ax, contrast_name) in enumerate(zip(axes, contrasts_order)):
        sub_contrast = summary[summary["contrast"] == contrast_name]

        for y in range(len(outcomes_order)):
            ax.axhline(y + 0.5, color="0.88", linewidth=0.7, alpha=0.7, zorder=0)

        for outcome in outcomes_order:
            if outcome not in y_positions:
                continue
            y_base = y_positions[outcome]

            for ctype, gtype, _, color, y_offset in triplet_specs:
                mask = sub_contrast["contrast_type"] == ctype
                if gtype is None:
                    mask &= sub_contrast["graph_family"].isna()
                else:
                    mask &= sub_contrast["graph_family"] == gtype
                mask &= sub_contrast["outcome"] == outcome

                sub = sub_contrast[mask]
                if sub.empty:
                    continue

                row = sub.iloc[0]
                is_sig = bool(row["sig"])
                alpha = 1.0 if is_sig else 0.45
                marker_face = color if is_sig else "white"

                ax.errorbar(
                    row[est_col],
                    y_base + y_offset,
                    xerr=[[row[est_col] - row[lo_col]], [row[hi_col] - row[est_col]]],
                    fmt="o",
                    color=color,
                    markerfacecolor=marker_face,
                    markeredgecolor=color,
                    capsize=3,
                    markersize=3,
                    alpha=alpha,
                    linewidth=1.5,
                )

        ax.axvline(0, color="k", linestyle="--", alpha=0.35, linewidth=1)
        ax.spines[["top", "right"]].set_visible(False)
        panel_label = chr(ord("A") + panel_idx)
        ax.set_title(
            f"({panel_label}) {contrast_labels.get(contrast_name, contrast_name)}",
            fontsize=13,
        )
        ax.set_xlabel(
            "Cohen's d" if use_standardized else "Effect estimate", fontsize=12
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.5)
        ax.spines["bottom"].set_linewidth(1.5)
        ax.tick_params(width=1.3, length=5)

        # major ticks
        ax.tick_params(axis="both", which="major", width=1.3, length=5)

        # minor ticks every 0.5 on x-axis
        ax.xaxis.set_minor_locator(MultipleLocator(0.5))
        ax.tick_params(axis="x", which="minor", width=1.0, length=3)

    axes[0].set_yticks(range(len(outcomes_order)))
    axes[0].set_yticklabels(
        [COL_PRETTY_NAMES.get(o, o) for o in outcomes_order], fontsize=12
    )

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color=color,
            markerfacecolor=color,
            markeredgecolor=color,
            linestyle="None",
            markersize=6,
            label=label,
        )
        for _, _, label, color, _ in triplet_specs
    ]
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=False,
        fontsize=12,
        bbox_to_anchor=(0.5, -0.07),
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

In [ ]:
contrasts_latex = pl.from_pandas(
    contrasts[contrasts["contrast_type"].isin(["marginal", "within", "interaction"])]
)


#### Marginal Effects

In [ ]:
lfs = (
    contrasts_latex.filter(pl.col("contrast_type") == "marginal")
    .with_columns(
        pl.col("outcome")
        .map_elements(lambda x: COL_PRETTY_NAMES.get(x, x), return_dtype=pl.String)
        .alias("Outcome")
    )
    .with_columns(
        pl.col("contrast")
        .map_elements(lambda x: contrast_labels.get(x, x), return_dtype=pl.String)
        .alias("Contrast")
    )
    .select(
        pl.col("Outcome"),
        pl.col("Contrast"),
        pl.col("cohens_d").alias("Cohen's d"),
        pl.col("cohens_d_lower").alias("d CI lower"),
        pl.col("cohens_d_upper").alias("d CI upper"),
        pl.col("SE"),
        pl.col("t"),
        pl.col("p").map_elements(fmt_p, return_dtype=pl.String).alias("p"),
        pl.col("df").round(0).cast(pl.Int32).alias("df"),
        pl.when(pl.col("sig")).then(pl.lit("*")).otherwise(pl.lit("")).alias("Sig"),
    )
).to_pandas()
lfs = fmt_contrast_cols(lfs)
lfs["Contrast"] = pd.Categorical(
    lfs["Contrast"], categories=CONTRAST_ORDER, ordered=True
)
lfs["Outcome"] = pd.Categorical(lfs["Outcome"], categories=OUTCOME_ORDER, ordered=True)
lfs = lfs.sort_values(["Outcome", "Contrast"]).set_index(["Outcome", "Contrast"])

marginal_latex = lfs.to_latex(
    index=True,
    column_format=latex_cols(lfs),
    caption="Marginal contrasts on key outcomes.",
    multicolumn=True,
    multirow=True,
    escape=False,
    label="tab:marginal-contrasts",
)

marginal_latex = marginal_latex.replace("\\multirow[t]{", "\\multirow[c]{")

print(marginal_latex)

### Conditional and Interaction Effects

In [ ]:
# Map graph_family: real values use graph_labels, nulls (interaction rows) become "Between"
network_expr = (
    pl.when(pl.col("contrast_type") == "interaction")
    .then(pl.lit("Between"))
    .otherwise(
        pl.col("graph_family").replace_strict(
            graph_labels, default=pl.col("graph_family")
        )
    )
    .alias("Network")
)

lfs = (
    contrasts_latex.filter(pl.col("contrast_type").is_in(["within", "interaction"]))
    .with_columns(
        pl.col("outcome")
        .map_elements(lambda x: COL_PRETTY_NAMES.get(x, x), return_dtype=pl.String)
        .alias("Outcome"),
        pl.col("contrast")
        .map_elements(lambda x: contrast_labels.get(x, x), return_dtype=pl.String)
        .alias("Contrast"),
        network_expr,
    )
    .select(
        "Outcome",
        "Contrast",
        "Network",
        pl.col("cohens_d").alias("Cohen's d"),
        pl.col("cohens_d_lower").alias("d CI lower"),
        pl.col("cohens_d_upper").alias("d CI upper"),
        pl.col("SE"),
        pl.col("t"),
        pl.col("p").map_elements(fmt_p, return_dtype=pl.String).alias("p"),
        pl.col("p_holm")
        .map_elements(fmt_p, return_dtype=pl.String)
        .fill_null("")
        .alias("p Holm"),
        pl.col("df").round(0).cast(pl.Int32).alias("df"),
        # Between rows: Holm across the 12 interaction tests of the level (E2/E3); ER/WS rows: unadjusted
        pl.when(
            pl.when(pl.col("contrast_type") == "interaction")
            .then(pl.col("p_holm") < 0.05)
            .otherwise(pl.col("sig"))
        )
        .then(pl.lit("*"))
        .otherwise(pl.lit(""))
        .alias("Sig"),
    )
).to_pandas()
lfs = fmt_contrast_cols(lfs)

lfs["Contrast"] = pd.Categorical(
    lfs["Contrast"], categories=CONTRAST_ORDER, ordered=True
)
lfs["Outcome"] = pd.Categorical(lfs["Outcome"], categories=OUTCOME_ORDER, ordered=True)
lfs["Network"] = pd.Categorical(
    lfs["Network"], categories=["ER", "WS", "Between"], ordered=True
)

# Group by network so each contrast's ER/WS/Between rows sit together per outcome
lfs = lfs.sort_values(["Outcome", "Network", "Contrast"]).set_index(
    ["Outcome", "Network", "Contrast"]
)

cond_latex = lfs.to_latex(
    index=True,
    column_format=latex_cols(lfs),
    caption="Within-network and between-network contrasts on key outcomes. "
    "The \\emph{Between} row reports the graph family$\\times$contrast interaction (ER$-$WS); "
    "its $p_{\\text{Holm}}$ and Sig are Holm-adjusted across the 12 interaction tests within the "
    "agent or the run level. ER and WS rows are unadjusted.",
    multicolumn=True,
    multirow=True,
    escape=False,
    label="tab:within-between-contrasts",
)

cond_latex = cond_latex.replace("\\multirow[t]{", "\\multirow[c]{")

print(cond_latex)

In [ ]:
fig = plot_contrast_forest(
    contrasts,
    outcomes_order=AGENT_COLS + RUN_COLS,
    use_standardized=True,
    figsize=(11, 3),
)
fig.savefig(
    result_path / "contrasts" / "contrast_forest.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
def plot_family_difference_forest(
    summary,
    contrasts_order=None,
    outcomes_order=None,
    use_standardized=True,
    figsize=(11, 3),
):
    """Forest plot of the family x contrast interaction (ER - WS), one panel per contrast.

    Same layout as `plot_contrast_forest`: one panel per contrast, one row per outcome.
    Each row shows the ER - WS difference of that contrast with its 95% CI. Filled
    markers and stars use the Holm-adjusted p (`p_holm`) written by E2 (agent level)
    and E3 (run level); the bars are unadjusted 95% CIs.

    Args:
        summary: Combined contrast summary (E2 + E3 CSVs).
        contrasts_order: Ordered list of contrasts to display as panels.
        outcomes_order: Ordered list of outcomes to display on the y-axis.
        use_standardized: Whether to use standardized estimates/CIs.
        figsize: Matplotlib figure size.

    Returns:
        Matplotlib figure.
    """
    if contrasts_order is None:
        contrasts_order = ["role_eff", "model_eff", "alignment", "dissociation"]
    if outcomes_order is None:
        outcomes_order = summary["outcome"].unique().tolist()

    contrast_labels = {
        "role_eff": "Role Effect",
        "model_eff": "Specialization Effect",
        "alignment": "Role-Specialization Effect",
        "dissociation": "Composition Effect",
    }

    est_col = "cohens_d" if use_standardized else "estimate"
    lo_col = "cohens_d_lower" if use_standardized else "lower"
    hi_col = "cohens_d_upper" if use_standardized else "upper"

    def stars(p):
        if pd.isna(p):
            return ""
        return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

    ix = summary[summary["contrast_type"] == "interaction"]

    fig, axes = plt.subplots(
        1,
        len(contrasts_order),
        figsize=figsize,
        sharey=True,
        sharex=True,
        squeeze=False,
    )
    axes = axes[0]

    y_positions = {o: i for i, o in enumerate(outcomes_order)}

    for panel_idx, (ax, contrast_name) in enumerate(zip(axes, contrasts_order)):
        sub_contrast = ix[ix["contrast"] == contrast_name]

        for y in range(len(outcomes_order)):
            ax.axhline(y + 0.5, color="0.88", linewidth=0.7, alpha=0.7, zorder=0)

        for outcome in outcomes_order:
            sub = sub_contrast[sub_contrast["outcome"] == outcome]
            if sub.empty:
                continue
            row = sub.iloc[0]

            is_sig = row["p_holm"] < 0.05
            color = OKABE_ITO["black"] if is_sig else "0.55"

            ax.errorbar(
                row[est_col],
                y_positions[outcome],
                xerr=[[row[est_col] - row[lo_col]], [row[hi_col] - row[est_col]]],
                fmt="o",
                color=color,
                markerfacecolor=color if is_sig else "white",
                markeredgecolor=color,
                capsize=3,
                markersize=4,
                linewidth=1.5,
            )
            ax.annotate(
                stars(row["p_holm"]),
                (row[hi_col], y_positions[outcome]),
                xytext=(3, -1),
                textcoords="offset points",
                va="center",
                fontsize=11,
            )

        ax.axvline(0, color="k", linestyle="--", alpha=0.35, linewidth=1)
        panel_label = chr(ord("A") + panel_idx)
        ax.set_title(
            f"({panel_label}) {contrast_labels.get(contrast_name, contrast_name)}",
            fontsize=13,
        )
        ax.set_xlabel(
            "Cohen's d (ER − WS)" if use_standardized else "Effect estimate (ER − WS)",
            fontsize=12,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.5)
        ax.spines["bottom"].set_linewidth(1.5)
        ax.tick_params(axis="both", which="major", width=1.3, length=5)

        ax.xaxis.set_minor_locator(MultipleLocator(0.1))
        ax.tick_params(axis="x", which="minor", width=1.0, length=3)

    axes[0].set_yticks(range(len(outcomes_order)))
    axes[0].set_yticklabels(
        [COL_PRETTY_NAMES.get(o, o) for o in outcomes_order], fontsize=12
    )

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color=OKABE_ITO["black"],
            linestyle="None",
            markersize=6,
            label="Holm p < .05",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="0.55",
            markerfacecolor="white",
            linestyle="None",
            markersize=6,
            label="n.s. (Holm)",
        ),
    ]
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=2,
        frameon=False,
        fontsize=12,
        bbox_to_anchor=(0.5, -0.07),
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig


fig = plot_family_difference_forest(
    contrasts,
    outcomes_order=AGENT_COLS + RUN_COLS,
    use_standardized=True,
    figsize=(11, 3),
)
fig.savefig(
    result_path / "contrasts" / "family_difference_forest.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Influence (ICC)

In [ ]:
ICC_SCENARIO_NAMES = {
    "base_llms": "I. Baseline",
    "random_roles": "II. Random Roles",
    "random_experts": "III. Random Specialists",
    "experts": "IV. Matched Specialists",
}

icc_path = result_path / "aggregated" / "agent"
icc_df = pd.read_csv(icc_path / "influence_out_icc_proj_contrast.csv")
icc_df = icc_df.set_index("run_scenario").loc[list(ICC_SCENARIO_NAMES)]


def fmt_p_lrt(p: float) -> str:
    return "<0.001" if p < 0.001 else f"{p:.2f}"


ICC_PCT_COLS = [
    "icc_agent",
    "icc_agent_lo",
    "icc_agent_hi",
    "icc_graph",
    "icc_graph_lo",
    "icc_graph_hi",
]
icc_pct = 100 * icc_df[ICC_PCT_COLS]
icc_pct["all"] = "all"
fmt_cols(icc_pct, ICC_PCT_COLS[:3], base=2, by="all")
fmt_cols(icc_pct, ICC_PCT_COLS[3:], base=2, by="all")
icc_pct = icc_pct[ICC_PCT_COLS]

rows = []
for scenario, r in icc_df.iterrows():
    rows.append(
        f"{ICC_SCENARIO_NAMES[scenario]} & "
        + " & ".join(icc_pct.loc[scenario])
        + f" & {fmt_p_lrt(r['lrt_p'])} \\\\"
    )

icc_latex = "\n".join(
    [
        r"\begin{table}",
        r"\centering",
        r"\begin{tabular}{lccc|ccc|r}",
        r"\toprule",
        r"  & \multicolumn{3}{c}{Agent-level ICC (95\% CI)} & \multicolumn{3}{c}{Network-level ICC (95\% CI)} &  \\",
        r"Scenario & ICC & Lower & Upper & ICC & Lower & Upper & RLRT $p$ \\",
        r"\midrule",
        *rows,
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{}",
        r"\label{tab:var-decomp}",
        r"\end{table}",
    ]
)
print(icc_latex)